## In this script,
- I will fetch image of specified PUMA
- I will take imagery of centeroid and jitter around it (just for testing)
- I will featurize the image with MOSAIKS
- and add to CPS dataset

In [1]:
import mosaiks 
print(mosaiks.__file__)

/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/mosaiks/__init__.py


In [2]:
import os
# ---- env flags (must be set BEFORE importing geopandas/dask) ----
os.environ["USE_PYGEOS"] = "0"
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "True"
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TARGET"] = "0.6"
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__SPILL"]  = "0.7"
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__PAUSE"]  = "0.8"
os.environ["DASK_DISTRIBUTED__WORKER__MEMORY__TERMINATE"] = "0.95"

import pandas as pd
import numpy as np
from mosaiks import get_features
import geopandas as gpd


os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')

In [3]:
# read cleaned_with_nearest_college datasets
cps14 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps14_with_nearest_college.parquet")
cps13 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps13_with_nearest_college.parquet")
cps12 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps12_with_nearest_college.parquet")

In [4]:
os.environ['USE_PYGEOS'] = '0'
from shapely.geometry import Point
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

# --- 1) US のPUMA shapefileを読む ---
shp_dir = Path("dataset/sharpfiles")
shp_files = list(shp_dir.glob("**/tl_2025_*_puma20.shp"))
print("Length:", len(shp_files))

gdfs = []
for f in shp_files:
    gdf = gpd.read_file(f, engine='pyogrio')
    gdfs.append(gdf)

puma_all_shp = pd.concat(gdfs, ignore_index=True)
puma_all_shp = gpd.GeoDataFrame(puma_all_shp, geometry="geometry", crs=gdfs[0].crs)

# 念のため key を作る
puma_all_shp["state_key"] = puma_all_shp["STATEFP20"].astype(str).str.zfill(2)
puma_all_shp["puma_key"] = puma_all_shp["PUMACE20"].astype(str).str.zfill(5)
puma_all_shp["state_puma"] = puma_all_shp["state_key"] + "_" + puma_all_shp["puma_key"]


# --- 2) ポリゴン内ランダム点サンプル ---
def sample_points_in_polygon(poly, n=200, seed=0, max_tries=200000):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = poly.bounds
    pts = []
    tries = 0

    while len(pts) < n and tries < max_tries:
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        p = Point(x, y)
        if poly.covers(p):
            pts.append(p)
        tries += 1

    if len(pts) < n:
        raise RuntimeError(f"Only sampled {len(pts)}/{n} points; polygon may be tiny/invalid.")

    return np.array([(pt.y, pt.x) for pt in pts])  # (lat, lon)


# --- 3) 全PUMAに拡張 ---
N_PER_PUMA = 5

all_lats = []
all_lons = []
all_puma = []

for idx, row in puma_all_shp.iterrows():
    code = row["state_puma"]
    poly = row.geometry

    if not poly.is_valid:
        poly = poly.buffer(0)

    latlon = sample_points_in_polygon(poly, n=N_PER_PUMA, seed=42 + idx)

    lats = latlon[:, 0]
    lons = latlon[:, 1]

    all_lats.append(lats)
    all_lons.append(lons)
    all_puma.append(np.repeat(code, N_PER_PUMA))

all_lats = np.concatenate(all_lats)
all_lons = np.concatenate(all_lons)
all_puma = np.concatenate(all_puma)

print("Total sampled points:", len(all_lats))
print("Unique PUMAs:", len(np.unique(all_puma)))

#save selected points to dataframe for reproductivity
sample_points = pd.DataFrame({
    "lat": all_lats,
    "lon": all_lons,
    "state_puma": all_puma
})
sample_points[["STATE", "PUMA"]] = sample_points["state_puma"].str.split("_", expand=True)
sample_points.to_parquet("dataset/sample_points_puma.parquet", index=False)


Length: 49
Total sampled points: 12235
Unique PUMAs: 2447


In [6]:
all_puma
print([all_lats[0], all_lons[0]])

[np.float64(44.64695697558939), np.float64(-71.21401538371171)]


In [11]:
import os, numpy as np

# 1) dask config の KeyError 対策（環境変数で確実に入れる）
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "True"

import dask
dask.config.set({"dataframe.query-planning": True})

# 2) dask client を明示（Macで安定）
from dask.distributed import Client
client = Client(processes=True, n_workers=4, threads_per_worker=1)  # 好きに調整OK
print(client)

<Client: 'tcp://127.0.0.1:61161' processes=4 threads=4, memory=16.00 GiB>


In [12]:
np.random.seed(2026)

In [ ]:
#テスト用１座標
feat = get_features(
    [40.7128],
    [-74.0060],
    datetime=["2020-01-01", "2020-12-31"],
    image_width=256,
    image_composite_method="all",
    parallelize=False,
    dask_chunksize=5,
    dask_n_workers=2,
    dask_threads_per_worker=1
)
print(feat)
print(feat.shape)

sample_lats = [40.7128, 36.1627, 34.0522, 41.8781, 29.7604]
sample_lons = [-74.0060, -86.7816, -118.2437, -87.6298, -95.3698]

feat = get_features(
    sample_lats,
    sample_lons,
    datetime=["2020-01-01", "2020-12-31"],
    image_width=64,
    image_composite_method="least_cloudy",
    image_dtype="float64",
    parallelize=False,
    dask_chunksize=5
)

print(feat.shape)
print(feat.iloc[:, :10])

/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse


(5, 4001)
   mosaiks_0  mosaiks_1  mosaiks_2  mosaiks_3  mosaiks_4  mosaiks_5  \
0        0.0   0.187355   0.000000        0.0   0.000048   0.762548   
1        0.0   1.034896   0.000000        0.0   0.000000   2.571148   
2        0.0   0.732990   0.000055        0.0   0.000015   1.944061   
3        0.0   0.311058   0.000124        0.0   0.000000   1.108544   
4        0.0   0.405739   0.000537        0.0   0.000279   1.451286   

   mosaiks_6  mosaiks_7  mosaiks_8  mosaiks_9  
0   0.000043        0.0        0.0   0.093820  
1   0.000000        0.0        0.0   0.134762  
2   0.000000        0.0        0.0   0.166510  
3   0.000000        0.0        0.0   0.139503  
4   0.000031        0.0        0.0   0.126768  


In [19]:
feat.columns

Index(['mosaiks_0', 'mosaiks_1', 'mosaiks_2', 'mosaiks_3', 'mosaiks_4',
       'mosaiks_5', 'mosaiks_6', 'mosaiks_7', 'mosaiks_8', 'mosaiks_9',
       ...
       'mosaiks_3991', 'mosaiks_3992', 'mosaiks_3993', 'mosaiks_3994',
       'mosaiks_3995', 'mosaiks_3996', 'mosaiks_3997', 'mosaiks_3998',
       'mosaiks_3999', 'stac_id'],
      dtype='object', length=4001)

In [13]:
# extract features for all sampled points and average within PUMA

from pathlib import Path

batch_size = 100
out_dir = Path("dataset/mosaiks_batches")
out_dir.mkdir(parents=True, exist_ok=True)

failed_batches = []

for start in range(0, len(all_lats), batch_size):
    end = min(start + batch_size, len(all_lats))
    print(f"processing {start}:{end}")

    batch_lats = all_lats[start:end].tolist()
    batch_lons = all_lons[start:end].tolist()

    try:
        feat = get_features(
            batch_lats,
            batch_lons,
            datetime=["2018-01-01", "2022-12-31"],
            image_width=64,
            image_composite_method="least_cloudy",
            parallelize=False
        )

        feat["orig_index"] = list(range(start, start + len(feat)))

        feature_cols = [c for c in feat.columns if c.startswith("mosaiks_")]
        bad_mask = feat[feature_cols].isna().all(axis=1)

        feat.to_parquet(out_dir / f"feat_{start:05d}_{end:05d}.parquet", index=False)

        if bad_mask.any():
            bad_idx = feat.loc[bad_mask, "orig_index"]
            pd.DataFrame({"orig_index": bad_idx}).to_parquet(
                out_dir / f"failed_points_{start:05d}_{end:05d}.parquet",
                index=False
            )

    except Exception as e:
        print(f"batch failed {start}:{end} -> {e}")
        failed_batches.append({"start": start, "end": end, "error": str(e)})

pd.DataFrame(failed_batches).to_csv(out_dir / "failed_batches.csv", index=False)



processing 0:100


/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse
/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/dask/array/chunk_types.py:131: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.sparse


processing 100:200


processing 200:300
processing 300:400
processing 400:500
processing 500:600
processing 600:700
processing 700:800
processing 800:900
processing 900:1000
processing 1000:1100
processing 1100:1200
processing 1200:1300
processing 1300:1400


processing 1400:1500
processing 1500:1600
processing 1600:1700
processing 1700:1800
processing 1800:1900


...

processing 11700:11800


processing 11800:11900


processing 11900:12000


processing 12000:12100


processing 12100:12200
processing 12200:12235


In [18]:
print(sample_points.columns)

Index(['lat', 'lon', 'state_puma', 'STATE', 'PUMA'], dtype='object')


In [29]:
#combine all files 

from pathlib import Path
import pandas as pd

# read sample data and img feat data
sample_points = pd.read_parquet("dataset/sample_points_puma.parquet")
files = sorted(Path("dataset/mosaiks_batches").glob("feat_*.parquet"))

# combine all img feat
dfs = [pd.read_parquet(f) for f in files]
feat_all = pd.concat(dfs, ignore_index=True)
feat_all = feat_all.sort_values("orig_index").reset_index(drop=True)

# sample_points と順序をそろえて point_id を付与
sample_points = sample_points.reset_index(drop=True)
sample_points["point_id"] = sample_points.index
feat_all["point_id"] = feat_all.index

# 安全チェック
assert len(sample_points) == len(feat_all), "sample_points と feat_all の行数が一致しません"

# sample point 情報を feat に付与
feat_all = feat_all.merge(sample_points, on="point_id", how="left")

# average over PUMA
feature_cols = [c for c in feat_all.columns if c.startswith("mosaiks_")]

feat_puma = (
    feat_all
    .groupby(["STATE", "PUMA"])[feature_cols]
    .mean()
    .reset_index()
)

# PUMAごとに1行か確認
assert feat_puma.duplicated(subset=["STATE", "PUMA"]).sum() == 0

# merge to CPS
cps14 = cps14.merge(feat_puma, on=["STATE", "PUMA"], how="left")
cps13 = cps13.merge(feat_puma, on=["STATE", "PUMA"], how="left")
cps12 = cps12.merge(feat_puma, on=["STATE", "PUMA"], how="left")


In [37]:
#save
cps14.to_parquet("dataset/cleaned_dataset/cleaned_cps14_with_nearest_college_imgfeat.parquet")
cps13.to_parquet("dataset/cleaned_dataset/cleaned_cps13_with_nearest_college_imgfeat.parquet")
cps12.to_parquet("dataset/cleaned_dataset/cleaned_cps12_with_nearest_college_imgfeat.parquet")

In [31]:
cps14.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place',
       ...
       'mosaiks_3990', 'mosaiks_3991', 'mosaiks_3992', 'mosaiks_3993',
       'mosaiks_3994', 'mosaiks_3995', 'mosaiks_3996', 'mosaiks_3997',
       'mosaiks_3998', 'mosaiks_3999'],
      dtype='object', length=4024)

In [35]:
cps14.shape

(148770, 4024)

In [32]:
len(all_puma)

12235

In [33]:
feat_all.shape

(12235, 4008)

In [34]:
feat_all.isna().mean().mean()

np.float64(0.08116174679655809)